# Cat's Card Generation

Another thing which is important for character generation is to generate cat's stats and abilities based on the context.

## Setup

In [24]:
import json
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(dotenv_path="../.env")

True

## Prompt Engineering

The planned flow for character generation is the following: 
- a user inputs their cat's name, photo and personality description

- cat's breed, colors ratio are extracted from photo via ML

- all of the above is sent to an LLM which generates a character that fits within the game's rules
  - stats (hp, mp, attack, defense, speed)
  - abilities (name, description, type, damage, cooldown, mana cost)
- the generated character is stored in the database

For this we use **Gemini API** (`gemini-2.5-flash`), which is capable of returning structured JSON output. The model receives context from all prior pipeline steps (breed from notebook 01, colors from notebook 02) plus the user-provided cat name and optional personality description.

The client reads gemini api key from the environment by default. (example is in the repo)

In [25]:
gemini_api = os.getenv('GEMINI_API_KEY')

client = genai.Client(api_key=gemini_api)

### Building the prompt

The prompt has to be explicit about the stat bounds so the model respects the game balance.

In [26]:
def build_prompt(cat_name: str, breed: str, colors: list[str], personality: str | None = None,) -> str:
    personality_clause = ""
    if personality:
        personality_clause = f"\nThe cat's personality: {personality}\n"

    return f"""You are a game designer for a cat-themed roguelike. Generate a playable character card.

Cat name: {cat_name}
Breed: {breed}
Fur colors: {colors}
{personality_clause}
Return ONLY valid JSON with these exact fields:
- "name": string (the cat's name)
- "class": "STRENGTH" | "AGILITY" | "INTELLIGENCE"
- "max_hp": int (30-200)
- "dmg": int (5-50)
- "defence": int (3-40)
- "spd": int (5-50)
- "max_mana": int (50-200)
- "lore": string (1-2 sentences of flavour text)
- "image_prompt": string (a concise subject description for an image generation model — cat breed, colors, class, mood — NO art style instructions, just the subject)
- "abilities": array of exactly 4 objects, exactly 1 with is_special: true
  each: {{"name": str, "dmg": int, "type": "DMG" | "HEAL" | "SHIELD" | "STEAL" | "AOE" | "COUNTER" | "TRUE_DMG", "effect": null | "STUN" | "SILENCE" | "BLEED" | "BURN" | "BLIND" | "SLOW" | "TAUNT" | "REGEN", "cooldown": int (0-5), "mana_cost": int (0-100), "lore": str, "is_special": bool, "description": str}}"""

### Testing the prompt

Let's test with a sample cat. The breed and colors come from notebooks 01 and 02, the name and personality come from the user.

In [27]:
prompt = build_prompt(
    cat_name="Sir Whiskers",
    breed="Siamese",
    colors=["#C0A080", "#8B6F47", "#F5E6D3"],
    personality="Aloof but secretly affectionate, loves to nap in sunbeams.",
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)

card = json.loads(response.text)
card

{'name': 'Sir Whiskers',
 'class': 'AGILITY',
 'max_hp': 80,
 'dmg': 25,
 'defence': 15,
 'spd': 45,
 'max_mana': 100,
 'lore': "Sir Whiskers appears to be lost in thought, often found napping in the warmest sunbeam, yet a keen eye might catch a flicker of affection in his gaze. Despite his seemingly lazy demeanor, he's always ready to spring into action when duty calls.",
 'image_prompt': 'Siamese cat, #C0A080, #8B6F47, #F5E6D3 fur, AGILITY class, aloof yet secretly affectionate, napping in a sunbeam',
 'abilities': [{'name': 'Quick Paw Strike',
   'dmg': 15,
   'type': 'DMG',
   'effect': None,
   'cooldown': 0,
   'mana_cost': 0,
   'lore': 'A swift, precise tap with a padded paw, executed with minimal effort.',
   'is_special': False,
   'description': 'Deals 15 damage to a single enemy.'},
  {'name': 'Sunbeam Dash',
   'dmg': 5,
   'type': 'DMG',
   'effect': 'BLIND',
   'cooldown': 2,
   'mana_cost': 20,
   'lore': 'Sir Whiskers vanishes and reappears in a flash of light, momenta

### Validating the output

The model is guided by the prompt, but we must validate server-side to enforce game balance. Every field is checked against the spec bounds.

In [28]:
def validate_card(card: dict) -> list[str]:
    errors: list[str] = []

    if not (30 <= card.get("max_hp", 0) <= 200):
        errors.append(f"max_hp out of bounds: {card.get('max_hp')}")
    if not (5 <= card.get("dmg", 0) <= 50):
        errors.append(f"dmg out of bounds: {card.get('dmg')}")
    if not (3 <= card.get("defence", 0) <= 40):
        errors.append(f"defence out of bounds: {card.get('defence')}")
    if not (5 <= card.get("spd", 0) <= 50):
        errors.append(f"spd out of bounds: {card.get('spd')}")
    if not (50 <= card.get("max_mana", 0) <= 200):
        errors.append(f"max_mana out of bounds: {card.get('max_mana')}")

    abilities = card.get("abilities", [])
    if len(abilities) != 4:
        errors.append(f"Expected 4 abilities, got {len(abilities)}")
    else:
        specials = [a for a in abilities if a.get("is_special")]
        if len(specials) != 1:
            errors.append(f"Expected 1 special ability, got {len(specials)}")

    for a in abilities:
        if not (0 <= a.get("mana_cost", -1) <= 100):
            errors.append(f"mana_cost out of bounds for '{a.get('name')}': {a.get('mana_cost')}")
        if not (0 <= a.get("cooldown", -1) <= 5):
            errors.append(f"cooldown out of bounds for '{a.get('name')}': {a.get('cooldown')}")

    return errors


errors = validate_card(card)
print("Validation errors:", errors if errors else "None ✨")

Validation errors: None ✨


In [29]:
if not errors:
    print(json.dumps(card, indent=2))

{
  "name": "Sir Whiskers",
  "class": "AGILITY",
  "max_hp": 80,
  "dmg": 25,
  "defence": 15,
  "spd": 45,
  "max_mana": 100,
  "lore": "Sir Whiskers appears to be lost in thought, often found napping in the warmest sunbeam, yet a keen eye might catch a flicker of affection in his gaze. Despite his seemingly lazy demeanor, he's always ready to spring into action when duty calls.",
  "image_prompt": "Siamese cat, #C0A080, #8B6F47, #F5E6D3 fur, AGILITY class, aloof yet secretly affectionate, napping in a sunbeam",
  "abilities": [
    {
      "name": "Quick Paw Strike",
      "dmg": 15,
      "type": "DMG",
      "effect": null,
      "cooldown": 0,
      "mana_cost": 0,
      "lore": "A swift, precise tap with a padded paw, executed with minimal effort.",
      "is_special": false,
      "description": "Deals 15 damage to a single enemy."
    },
    {
      "name": "Sunbeam Dash",
      "dmg": 5,
      "type": "DMG",
      "effect": "BLIND",
      "cooldown": 2,
      "mana_cost": 20,

### Wrapping up

The prompt + validation logic feeds directly into `backend/services/card_generator.py` as the `generate_card()` implementation. The notebook serves as a playground for iterating on the prompt — tweak the wording or stat bounds here, then copy the final version to the backend service.